# Soup Fusion — run the merged candidate on a T4

**Goal:** an admitted median whole-command time **strictly below 33 seconds**. The 77-second reference and 33-second record were supplied by the user; their original logs were not available.

**This notebook has not been executed on a T4 yet.** The accompanying source passed 250 CPU/interpreter checks and all 79 official regressions. Offline compilation and CPU checks are not GPU performance evidence.

## Kaggle setup

1. Import this notebook into Kaggle.
2. Enable Internet and select **T4 GPU** hardware. The code exposes only GPU 0.
3. Use **Run All**. No team URL or source archive upload is needed: the checked source is embedded below.
4. Download the results archive at the end. It contains the actual comparison and the selected source.

The search evaluates four fixed source profiles in separate local snapshots: `fused`, `cached256`, `epilogues`, `streamed`. Every profile must pass native GPU tests and the unchanged official two-repeat protocol. It stops after a qualifying sub-target result; otherwise it retains the fastest admitted profile and reports that the target was not met.

This runs the public benchmark protocol. It does not submit a team entry, establish hidden-row quality, or certify a world record. All compiled caches start fresh in each official measured run.


## 1. Verify and unpack the embedded source


In [ ]:
import base64, hashlib, io, json, os, stat, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

# Ordinary Python source and validation records only: no weights or compiled caches.
ARCHIVE_SHA256 = "dd9f3c3922cc2e54eaf288f2bd220fdfb103b61511195af95c5a15106dece995"
SOURCE_ARCHIVE_B64 = (
    '...'
    'AJMUBgAAAA=='
)
payload = base64.b64decode(SOURCE_ARCHIVE_B64, validate=True)
assert hashlib.sha256(payload).hexdigest() == ARCHIVE_SHA256
working = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
run_tag = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
SOURCE_ROOT = working / ('soup_fusion_source_' + run_tag)
SOURCE_ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        name = PurePosixPath(item.filename)
        assert not name.is_absolute() and '..' not in name.parts
        assert not stat.S_ISLNK(item.external_attr >> 16)
    z.extractall(SOURCE_ROOT)
manifest = json.loads((SOURCE_ROOT / 'MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert hashlib.sha256((SOURCE_ROOT / name).read_bytes()).hexdigest() == expected, name
os.environ.update(CUDA_VISIBLE_DEVICES='0', CUDA_DEVICE_ORDER='PCI_BUS_ID',
                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
os.environ.pop('TRITON_INTERPRET', None)
os.environ.pop('SOUP_APPLY_SOLUTION', None)
os.environ.pop('PYTORCH_ALLOC_CONF', None)
SEARCH_OUT = working / ('soup_fusion_results_' + run_tag)
print('Verified source:', SOURCE_ROOT)
print('Source solution hash:', manifest['solution_sha256'])
print('Future results:', SEARCH_OUT)


## 2. Install the fixed stack before measurement


In [ ]:
import subprocess

# Keep the notebook parent free of a Torch/CUDA context. All GPU work runs in
# child processes which exit before the official measured commands.
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version',
                      '--format=csv,noheader'], text=True, capture_output=True, check=True)
print(gpu.stdout)
assert 'T4' in gpu.stdout, 'Select Kaggle T4 hardware before running the benchmark.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(SOURCE_ROOT / 'requirements.txt')],
               check=True)
print('Organizer package pins installed before measurement.')
print('Torch/Triton actual versions and native path compatibility are checked by readiness.')


## 3. Run native readiness and the official paired comparisons


In [ ]:
import signal

# No timing is accepted from this driver itself. The untouched official
# harness supplies each measured command time and sampled GPU peak.
sys.path.insert(0, str(SOURCE_ROOT / 'tools'))
from run_t4_readiness import _stop_child_tree

command = [sys.executable, str(SOURCE_ROOT / 'tools/run_search.py'),
           '--record-seconds', '33', '--out', str(SEARCH_OUT)]
print('Running:', ' '.join(command))
process = subprocess.Popen(command, cwd=SOURCE_ROOT, env=dict(os.environ),
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    SEARCH_RETURN_CODE = process.wait()
except KeyboardInterrupt:
    try:
        process.send_signal(signal.SIGINT)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        _stop_child_tree(process)
        process.wait()
    raise
except BaseException:
    _stop_child_tree(process)
    process.wait()
    raise

print('\nSearch exit code:', SEARCH_RETURN_CODE)
print('Continue to the next cell to inspect and download evidence, including failed candidates.')


## 4. Inspect and download the measured result


In [ ]:
from IPython.display import display, Markdown, FileLink

summary_path = SEARCH_OUT / 'search_summary.json'
assert summary_path.exists(), 'No search summary exists; inspect the earlier error.'
summary = json.loads(summary_path.read_text())
rows = ['| Profile | Status | Paired baseline s | Candidate median s | Peak MiB |',
        '|---|---|---:|---:|---:|']
for entry in summary['profiles']:
    values = entry.get('official_summary') or {}
    base = values.get('base') or {}
    solution = values.get('solution') or {}
    rows.append('| {} | {} | {} | {} | {} |'.format(
        entry['profile'], entry['status'], base.get('wall_s', '—'),
        solution.get('wall_s', '—'), solution.get('gpu_peak_mib', '—')))
display(Markdown('\n'.join(rows)))
print('Status:', summary['status'])
print('Verified target met:', summary['target_met_under_official_public_protocol'])
print('Fastest admitted source:', summary.get('best_source_directory'))
print('World record verified:', summary['world_record_verified'])

result_zip = working / ('Soup_Fusion_Results_' + run_tag + '.zip')
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(SEARCH_OUT.rglob('*')):
        if not path.is_file():
            continue
        relative = path.relative_to(SEARCH_OUT)
        if any(part in {'.git', '__pycache__', '.pytest_cache', 'triton_cache', 'inductor_cache'}
               for part in relative.parts):
            continue
        if path.suffix in {'.pyc', '.pyo', '.ptx', '.cubin', '.so'}:
            continue
        z.write(path, str(relative))
print('Evidence archive:', result_zip, '| bytes:', result_zip.stat().st_size)
display(FileLink(str(result_zip)))
display(FileLink(str(summary_path)))
if not summary['target_met_under_official_public_protocol']:
    display(Markdown('**The sub-33-second target has not been demonstrated.** '
                     'Use the measured logs and retained source for the next optimization pass.'))
else:
    display(Markdown('**A candidate met the sub-33-second target under the public protocol.** '
                     'Retain this exact source and evidence; hidden-jury admission and '
                     'comparison with the original record remain separate.'))
